In [1]:
# packages
import pandas as pd
from mod02_build_bot_predictor import train_model

### Define a function to extract predictions from the model

In [2]:
def predict_bot(df, model=None):
    """
    Predict whether each account is a bot (1) or human (0).
    """
    if model is None:
        model = train_model()

    preds = model.predict(df)
    return pd.Series(preds, index=df.index)

### Define a function to evaluate model error

In [3]:
def confusion_matrix_and_metrics(y_true, y_pred):
    """
    Computes confusion matrix and common error rates for binary classification.

    Assumes labels:
      0 = negative class
      1 = positive class

    Returns:
      dict with:
        tn, fp, fn, tp
        misclassification_rate
        false_positive_rate
        false_negative_rate
    """
    tn = fp = fn = tp = 0

    for yt, yp in zip(y_true, y_pred):
        if yt == 0 and yp == 0:
            tn += 1
        elif yt == 0 and yp == 1:
            fp += 1
        elif yt == 1 and yp == 0:
            fn += 1
        elif yt == 1 and yp == 1:
            tp += 1
        else:
            raise ValueError("Labels must be 0 or 1")

    total = tn + fp + fn + tp

    misclassification_rate = (fp + fn) / total if total > 0 else 0.0
    false_positive_rate = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    false_negative_rate = fn / (fn + tp) if (fn + tp) > 0 else 0.0

    return {
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "misclassification_rate": misclassification_rate,
        "false_positive_rate": false_positive_rate,
        "false_negative_rate": false_negative_rate,
    }


### Load the data

In [4]:
TRAIN_PATH = "mod02_data/train.csv"
train = pd.read_csv(TRAIN_PATH)

TEST_PATH = "mod02_data/test.csv"
test = pd.read_csv(TEST_PATH)

### Format the data by independent vs. dependent variables

In [5]:
X_train = train.drop(columns=["is_bot"])
y_train = train['is_bot']

X_test = test.drop(columns=["is_bot"])
y_test = test['is_bot']

### Build the model on training data

In [6]:
model = train_model(X_train, y_train)

### Get the model predictions on training and test data

In [7]:
y_pred_train = predict_bot(X_train, model)
y_pred_test = predict_bot(X_test, model)

### Check results on the training set (data used to build the model)

In [8]:
confusion_matrix_and_metrics(y_train, y_pred_train)

{'tp': 111,
 'tn': 2597,
 'fp': 40,
 'fn': 252,
 'misclassification_rate': 0.09733333333333333,
 'false_positive_rate': 0.015168752370117557,
 'false_negative_rate': 0.6942148760330579}

### Check results on the test set (new data not yet seen by the model)

In [9]:
confusion_matrix_and_metrics(y_test, y_pred_test)

{'tp': 31,
 'tn': 857,
 'fp': 17,
 'fn': 95,
 'misclassification_rate': 0.112,
 'false_positive_rate': 0.019450800915331808,
 'false_negative_rate': 0.753968253968254}

# Discussion Questions

### Based on the misclassification rate of your model, discuss your confidence in the ability to predict a bot. 

The misclassificication rate on the test set was 0.112, which means the model correctly identifies about 9 out of 10 accounts. However, the false negative rate was 0.754, which means the model is missing a large proportion of bot accounts; if we want to lower this false negative rate, it will likely cause an increase in our misclassification rate and false positive rates, as the model will be more "aggressive" in its predictions. So, while the model correctly identifies most accounts, in actuality, most of the bots are making it past the model.

### What are potential ramifications of false positives from the model?

For this model, a false positive represents a human which was predicted to be a bot. In this case, the false positive rate was 0.019 (for the test set), which means the model is correctly predicting most humans to be human. If we are a company providing a product, there are obvious costs in turning away potential customers or users after falsely determing them to be a bot. Depending on the product, it may be more beneficial to keep the false positive rate low, even if it leads to a increase of false negatives, if the chief goal is to have the maximum amount of customers.

### What are potential ramifications of false negatives from the model?

For this model, false negatives represent bots which the model determined to be human. The false negative rate was 0.754 (for the test set), which means the model is predcting most bots to be human. These bots are likely to be acting with some malicious intent or act in a way disruptive to the actual human users (for instance, a social media platform being flooded by advertisments by bots).